# Notebook 03 (R) — Importing Raw Clinical Data

**Module:** Reading Raw Data · **Pairs with:** `03_import_raw_data_SAS.sas`

Goal: read all seven ABC-01 raw files, get the **types** right, and run the six-point
inspection checklist from Module 04 — so you know exactly what must change before
these can become SDTM.

Reference: `../../data/raw_data_dictionary.md`

## 0. Setup

In [ ]:
library(readr)
library(dplyr)
library(tidyr)

datapath <- "/Volumes/D Drive/SDTM Training/Bootcamp/data"   # <-- EDIT if needed

## 1. Type guessing and the leading-zero trap

IDs like `SITEID` ("01") and `SUBJID` ("001") are all digits, so a reader may decide
they are **numbers** — and the leading zeros disappear. Let's check what actually
happens in R.

In [ ]:
dm_naive <- read_csv(file.path(datapath, "dm_raw.csv"), show_col_types = FALSE)

sapply(dm_naive[, c("STUDYID", "SITEID", "SUBJID")], class)
dm_naive |> select(STUDYID, SITEID, SUBJID, SEX, ARM) |> head(3)

**Good news:** `readr::read_csv()` is smart here — it notices the leading zeros and
keeps those columns as **character**, so `01` and `001` survive intact.

**But don't rely on it.** Base R's `read.csv()` makes the opposite choice on the same
file — and so does SAS's `PROC IMPORT`:

In [ ]:
# base R: the same column becomes an integer and the zeros are lost
dm_base <- read.csv(file.path(datapath, "dm_raw.csv"))
cat("read.csv  SITEID:", class(dm_base$SITEID), "->", dm_base$SITEID[1], "\n")
cat("read.csv  SUBJID:", class(dm_base$SUBJID), "->", dm_base$SUBJID[1], "\n")

> **Why this matters.** If the zeros are lost, `USUBJID` becomes `ABC-01-1-1` instead
> of `ABC-01-01-001` — wrong for every subject, in every domain.
>
> **The rule:** never leave ID types to luck. Declare them explicitly, as below. It
> states your intent, survives a change of tool, and protects you if a future extract
> has IDs that *don't* start with a zero (e.g. site `10`), where the guess would flip.

## 2. The controlled import

Declare the types you want with `col_types`. `col_character()` forces text;
`.default = col_guess()` lets readr guess the rest.

In [ ]:
dm_raw <- read_csv(
  file.path(datapath, "dm_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess())
)

dm_raw |> select(STUDYID, SITEID, SUBJID, SEX, ARM) |> head(3)   # 01 and 001 preserved

Now the other six. For **AE** and **CM** we read *everything* as character so the
raw date strings survive exactly as recorded — we want to SEE the mixed formats
before converting them.

In [ ]:
ds_raw <- read_csv(file.path(datapath, "ds_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess()))

ex_raw <- read_csv(file.path(datapath, "ex_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess()))

ae_raw <- read_csv(file.path(datapath, "ae_raw.csv"),
  col_types = cols(.default = col_character()))

cm_raw <- read_csv(file.path(datapath, "cm_raw.csv"),
  col_types = cols(.default = col_character()))

vs_raw <- read_csv(file.path(datapath, "vs_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess()))

lb_raw <- read_csv(file.path(datapath, "lb_raw.csv"),
  col_types = cols(SITEID = col_character(), SUBJID = col_character(), .default = col_guess()))

cat("all seven files loaded\n")

---
# The six-point inspection checklist

Run this on every unfamiliar dataset, every time.

## Check 1 — How many rows and columns?

Expected: dm 8 · ds 8 · ex 8 · ae 9 · cm 8 · vs 24 · lb 48. A wrong count means something
was silently dropped.

In [ ]:
tibble(
  dataset = c("dm_raw", "ds_raw", "ex_raw", "ae_raw", "cm_raw", "vs_raw", "lb_raw"),
  rows    = c(nrow(dm_raw), nrow(ds_raw), nrow(ex_raw), nrow(ae_raw), nrow(cm_raw), nrow(vs_raw), nrow(lb_raw)),
  cols    = c(ncol(dm_raw), ncol(ds_raw), ncol(ex_raw), ncol(ae_raw), ncol(cm_raw), ncol(vs_raw), ncol(lb_raw))
)

## Check 2 — What type is every column?

Confirm the ID columns are `<chr>`, not `<dbl>`.

In [ ]:
glimpse(dm_raw)

## Check 3 — How many distinct subjects?

`SUBJID` alone is **not** unique — subject `001` exists at both sites. Build a
site+subject key to count real people.

In [ ]:
dm_raw |>
  summarise(
    distinct_subjid = n_distinct(SUBJID),                       # 4  (misleading!)
    distinct_people = n_distinct(paste(SITEID, SUBJID))         # 8  (correct)
  )

Every subject in a child domain must exist in DM. An **orphan** is a data-integrity
error — `anti_join()` finds rows on the left with no match on the right.

In [ ]:
dm_keys <- dm_raw |> select(SITEID, SUBJID)

bind_rows(
  ae_raw |> distinct(SITEID, SUBJID) |> anti_join(dm_keys, by = c("SITEID","SUBJID")) |> mutate(domain = "AE"),
  vs_raw |> distinct(SITEID, SUBJID) |> anti_join(dm_keys, by = c("SITEID","SUBJID")) |> mutate(domain = "VS"),
  lb_raw |> distinct(SITEID, SUBJID) |> anti_join(dm_keys, by = c("SITEID","SUBJID")) |> mutate(domain = "LB")
)   # expect 0 rows

## Check 4 — Which columns have blanks, and what does each blank mean?

A blank is not automatically an error — but you must know *why* it is blank.

In [ ]:
# AE: a blank end date means the event was still ONGOING
ae_raw |> filter(is.na(AEENDT)) |> select(SITEID, SUBJID, AETERM, AESTDT, AEENDT)

In [ ]:
# VS: HEIGHT is collected at SCREENING only, so it is missing at later visits
vs_raw |>
  summarise(across(c(SYSBP, DIABP, PULSE, TEMP, HEIGHT, WEIGHT), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "variable", values_to = "n_missing")

## Check 5 — What are the distinct values?

For any column you will map to Controlled Terminology, look at **every** value.
You cannot map values you have not seen.

In [ ]:
dm_raw |> count(SEX)
dm_raw |> count(RACE)      # note the mixed case: White / asian / black or african american
dm_raw |> count(ARM)

The raw file also contains `"White "` with a **trailing space**. You won't see it
above, because `read_csv()` trims whitespace by default (`trim_ws = TRUE`). That is
usually what you want — but it's worth knowing the setting exists, and that the
untrimmed value is really in the file:

In [ ]:
read_csv(file.path(datapath, "dm_raw.csv"), trim_ws = FALSE, show_col_types = FALSE) |>
  count(RACE)   # now "White " appears as its own value

In [ ]:
ae_raw |> count(AESEV)     # mild / Mild / moderate / Moderate / severe
ae_raw |> count(AESER)     # 'No' and 'N' both appear
ae_raw |> count(AEOUT)

## Check 6 — What date formats appear?

More than one format in the same column is common — and dangerous.

In [ ]:
date_format <- function(x) {
  dplyr::case_when(
    is.na(x)               ~ "(blank)",
    grepl("/", x)          ~ "DD/MM/YYYY",
    grepl("[A-Za-z]", x)   ~ "DD-Mon-YYYY",
    TRUE                   ~ "ISO or other"
  )
}

ae_raw |> mutate(fmt = date_format(AESTDT)) |> count(fmt)

> ⚠️ **Ambiguity warning.** `01/03/2024` could be 1 March or 3 January — you cannot
> tell from the value alone. The data dictionary states these are `DD/MM/YYYY`.
> Never guess a date convention.

---
## What you now know

| Finding | What it means for mapping |
|---|---|
| `SUBJID` repeats across sites | must build `USUBJID` |
| `SEX` is 1/2 | apply Controlled Terminology |
| `RACE` has mixed case + trailing space | trim and upper-case before mapping |
| Two date formats in AE/CM | convert everything to ISO 8601 |
| Blank `AEENDT` | ongoing — leave null |
| `HEIGHT` missing after screening | not collected — leave null |
| VS is wide | must be transposed for SDTM |

That list is the gap analysis from Module 04 — and the work plan for the modules ahead.

## YOUR TURN — exercises

Solutions: `../../answer-keys/03_import_answers.md`

**Exercise 1.** Run the format check on `cm_raw$CMSTDT`. How many of each format?

**Exercise 2.** How many distinct subjects appear in `lb_raw`? Which DM subjects have
no lab data at all?

**Exercise 3.** List every distinct `LBTEST` with its record count. Do all tests appear
the same number of times?

**Exercise 4 (stretch).** `vs_raw` is wide. Without transposing it, work out how many
rows a tall SDTM VS dataset would have — count the non-missing measurements across
all six vital-sign columns. (You should get **128** — check against `../../data/sdtm/vs.csv`.)

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4 (stretch)